<a href="https://colab.research.google.com/github/rxphaelbihag/Linear-Programming/blob/main/Store_Location_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Store Location Optimization

DS115: End-of-Sem Project

## Description
In this project, you are an entrepreneur with a budget enough to open 2 convenience stores in downtown Davao. The idea is to find the best locations of only two convenience stores that would allow more access to your target clients. You are provided 7 locations to choose from and you are given the locations of your 5 target clients.

## Specifications
1. The coordinates of the 7 possible store locations are stored in a .CSV file.
2. The coordinates of the 5 clients are stored in a .CSV file.
3. Randomly pick two pairs of possible store locations.
4. Using p-center as the math model for this problem, determine which of the
two sets should you choose as locations for your stores
5. Map the clients and the winning set stores

## Calculations
### Libraries and CSV files
We use `pandas` as our main tool for data manipulation. The `cdist` class from `scipy` allows us to calculate eucledian distances. We use `random` to generate random numbers. We use `folium` to generate the map of the final answer.

In [27]:
import pandas as pd
from scipy.spatial.distance import cdist
import random
import folium

### Import the CSV Files

In [28]:
store_locations_csv = "/content/drive/MyDrive/2BSDS/DS 115/bihag_storelocs.csv"
client_locations_csv = "/content/drive/MyDrive/2BSDS/DS 115/client_locs.csv"

stores = pd.read_csv(store_locations_csv, index_col='store')
clients = pd.read_csv(client_locations_csv, index_col='client')

In [29]:
stores

,latitude,longitude
store,,
1,7.090138,125.606766
2,7.088724,125.630833
3,7.081656,125.612843
4,7.081869,125.621222
5,7.079220,125.625024
6,7.072015,125.602871
7,7.065656,125.610641


In [30]:
clients

,latitude,longitude
client,,
1,7.086733,125.621816
2,7.082687,125.615703
3,7.077230,125.604769
4,7.085151,125.607086
5,7.073291,125.611879


### Random Two Sets
We randomly pick two sets or two pairs of store locations. The store locations are indexed from 1 to 7 (Store 1 to Store 7). So, we use `random`'s `sample()` function to sample four random numbers and group them to two. This is to ensure that the two sets do not have the same stores.

In [31]:
numbers = random.sample(range(1, 8), 4)

# The sets in index form
set1 = [numbers[0], numbers[1]]
set2 = [numbers[2], numbers[3]]

### Construct Distance Matrix

We use a distance matrix to calculate the distance of each client to each store in the generated sets. This is so that it will be easier for calculations later.

In [32]:
client_coords = clients[['latitude', 'longitude']].values
store_coords = stores[['latitude', 'longitude']].values

# The sets in coordinates form
set1_coords = [store_coords[set1[0]-1], store_coords[set1[1]-1]]
set2_coords = [store_coords[set2[0]-1], store_coords[set2[1]-1]]

In [33]:
# Compute the pairwise distance matrices for the mutated coordinates
set1_dist_clients = cdist(client_coords, set1_coords, metric='euclidean')
set2_dist_clients = cdist(client_coords, set2_coords, metric='euclidean')

# Build the labeled DataFrames (Rows = Clients, Columns = Store IDs)
set1_dist_matrix = pd.DataFrame(
    set1_dist_clients,
    index=clients.index,  # Keeps rows labeled as client1 to client5
    columns=set1    # Sets columns to the active store IDs (e.g., [3, 7])
)

set2_dist_matrix = pd.DataFrame(
    set2_dist_clients,
    index=clients.index,  # Keeps rows labeled as client1 to client5
    columns=set2    # Sets columns to the active store IDs (e.g., [1, 4])
)

In [34]:
print("Set 1")
print(f"{'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*40)
print(f"Store {set1[0]:<1} | {set1_coords[0][0]:<10.10f} | {set1_coords[0][1]:<10.10f}")
print(f"Store {set1[1]:<1} | {set1_coords[1][0]:<10.10f} | {set1_coords[1][1]:<10.10f}")

print("\n\nSet 2")
print(f"{'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*40)
print(f"Store {set2[0]:<1} | {set2_coords[0][0]:<10.10f} | {set2_coords[0][1]:<10.10f}")
print(f"Store {set2[1]:<1} | {set2_coords[1][0]:<10.10f} | {set2_coords[1][1]:<10.10f}")

Set 1
Store # | Latitude     | Longitude      
----------------------------------------
Store 7 | 7.0656564747 | 125.6106411320
Store 1 | 7.0901382366 | 125.6067659888


Set 2
Store # | Latitude     | Longitude      
----------------------------------------
Store 5 | 7.0792195616 | 125.6250239650
Store 2 | 7.0887236013 | 125.6308328384


In [35]:
set1_dist_matrix

,7,1
client,,
1,0.023856,0.015430
2,0.017767,0.011636
3,0.012978,0.013062
4,0.019816,0.004997
5,0.007734,0.017606


In [36]:
set2_dist_matrix

,5,2
client,,
1,0.008170,0.009234
2,0.009945,0.016290
3,0.020352,0.028486
4,0.018893,0.024014
5,0.014420,0.024442


### Solution Evaluation
Next, we evaluate the distances of each of the clients to each of the stores in our two random sets. For each client, we look at the store it's closest to. After that, we look at the worst-case distance among those closest stores.

In [37]:
# Evaluating first and second set using mini distance matrices
"""
Stores the distances. Format: "Client 1": [<Distance>, <Closest Store>]
- Distance is in degrees
- Closest Store is either 0 if closest to the first store in the set, 1 if otherwise
"""
set1_dists = {}
set2_dists = {}

for client in range(1, 5+1):
    s1store1 = set1[0]    # index of store1 in set1
    s1store2 = set1[1]    # index of store2 in set1
    s2store1 = set2[0]    # index of store1 in set2
    s2store2 = set2[1]    # index of store2 in set2

    s1distto_s1 = set1_dist_matrix[s1store1][client] # dist of client to store1,set1
    s1distto_s2 = set1_dist_matrix[s1store2][client] # dist of client to store2,set1

    s2distto_s1 = set2_dist_matrix[s2store1][client] # dist of client to store1,set2
    s2distto_s2 = set2_dist_matrix[s2store2][client] # dist of client to store2,set2

    # find the minimum among the two distances for Set 1
    if s1distto_s1 > s1distto_s2:
        set1_dists[f"client{client}"] = (s1distto_s2, 1)
    else:
        set1_dists[f"client{client}"] = (s1distto_s1, 0)

    # find the minimum among the two distances for Set 2
    if s2distto_s1 > s2distto_s2:
        set2_dists[f"client{client}"] = (s2distto_s2, 1)
    else:
        set2_dists[f"client{client}"] = (s2distto_s1, 0)

In [38]:
# Print Set 1 results
print(f"{'Client':<10} | {'Distance':<12} | {'Store':<10}")
print("-" * 40)

for key, value in set1_dists.items():
    store_name = set1[value[1]]
    print(f"{key:<10} | {value[0]:<12.8f} | {store_name:<10}")

Client     | Distance     | Store     
----------------------------------------
client1    | 0.01543025   | 1         
client2    | 0.01163590   | 1         
client3    | 0.01297810   | 7         
client4    | 0.00499704   | 1         
client5    | 0.00773387   | 7         


In [39]:
# Print Set 2 results
print(f"{'Client':<10} | {'Distance':<12} | {'Store':<10}")
print("-" * 40)

for key, value in set2_dists.items():
    store_name = set2[value[1]]
    print(f"{key:<10} | {value[0]:<12.8f} | {store_name:<10}")

Client     | Distance     | Store     
----------------------------------------
client1    | 0.00816980   | 5         
client2    | 0.00994498   | 5         
client3    | 0.02035244   | 5         
client4    | 0.01889294   | 5         
client5    | 0.01442040   | 5         


In [40]:
# Evaluate which set is the best
if max(set1_dists.values())[0] > max(set2_dists.values())[0]:
    best = set2
    print("The best set is Set 2:", set2, "The max distance (deg) is", max(set2_dists.values())[0])
else:
    best = set1
    print("The best set is Set 1:", set1, "The max distance (deg) is", max(set1_dists.values())[0])

The best set is Set 1: [7, 1] The max distance (deg) is 0.015430246996441958


### Optimal Solution

Now that we have identified the best set. We plot them on a map to visualize it.

In [41]:
# Initialize map center (Somewhere in downtown Davao City)
mymap = folium.Map(location=[7.080506387226319, 125.6120560701777],
                   zoom_start=15)

# dictionary of client coordinates
clients_map = {}
for _ in range(len(client_coords)):
    clients_map[f"Client {_+1}"] = [client_coords[_][0], client_coords[_][1]]

# dictionary of store coordinates in the best set
best_set = {}
for _ in range(2):
    best_set[f"Store {best[_]}"] = [store_coords[best[_]-1][0],
                                    store_coords[best[_]-1][1]]

# to determine the max radius in the best set
best_dists = set2_dists if best == set2 else set1_dists
max_radius_degrees1 = max(best_dists.values())[0]

# Draws the clients on the map
for client, coords in clients_map.items():
    folium.Marker(
        location=coords,
        popup=client,
        icon=folium.Icon(color="blue", icon="person", prefix='fa')
    ).add_to(mymap)

# Draws the stores on the map
for store, coords in best_set.items():
    folium.Marker(
        location=coords,
        popup=store,
        icon=folium.Icon(color="red", icon="store", prefix='fa')
    ).add_to(mymap)

    # Draws the circles around the stores of the best set
    folium.Circle(
        location=coords,
        radius=max_radius_degrees1 * 111320, # Radius from deg to meters
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.2,
        popup="Coverage Area"
    ).add_to(mymap)

mymap

# Genetic Algorithm Application
Below is the extension of the optimization problem above by applying Genetic Algorithm to the solutions.

## Specifications
1. Apply the crossover operator on the 2 sets of locations. By using a single point crossover, randomly select a crossover point and generate two offspring by swapping the 1st part of parent1 with 1st part of parent2.   
2. Based on the two new offsprings (after crossover operator), apply single point mutation on each of the new offspring. For offspring1, randomly select a single mutation point and mutate the gene in that location by adding 0.01 (or 1 x 10-2) in that location. For offspring2, select another random mutation point (preferably different from the point chosen in offspring1 and mutate that gene as well by adding 0.001 or (1 x 10-3) in that location.
3. Calculate the fitness value of each of the new offspring.
4. Compare the performance of the two new offspring with the parents.

## Calculations

Let $S_i$ represent the $i$-th set, and let $store_{ij} = (\text{lat_{ij}, lon_{ij}})$ denote the $j$-th element of set $i$, where $i \in {1,2}$ and $j \in {1,2}$.

**Crossover Operator**

Let $\text{Set 1} = \text{Parent 1}$ and $\text{Set 2} = \text{Parent 2}$.
$$S_1 = \text{Parent 1}=(store_{11}, store_{12})$$
$$S_2 = \text{Parent 2}=(store_{21}, store_{22})$$
Executing crossover operator,
$$\text{Offspring 1}=(store_{21}, store_{12})$$
$$\text{Offspring 2}=(store_{11}, store_{22})$$

**Mutation Operator**

Let random integer $z$, be the corresponding mutation point index. If $z=1$, then
$$\text{Offspring 1} = (store_{21}, store_{12}+0.01)$$
$$\text{Offspring 2} = (store_{11}+0.001, store_{22})$$

### Crossover Operator

In [42]:
# Print the parents
print("-- Parents --")
print("Parent 1 (Set 1):", set1)
print("Parent 2 (Set 2):", set2)

# Execute crossover
print("\n## Crossover ##\n")
offspring1, offspring2 = [set2[0], set1[1]], [set1[0], set2[1]]
offspring1_coords = [store_coords[offspring1[0]-1], store_coords[offspring1[1]-1]]
offspring2_coords = [store_coords[offspring2[0]-1], store_coords[offspring2[1]-1]]

# Print the offspring
print("-- Offspring --")
print("Offsping 1:", offspring1)
print("Offsping 2:", offspring2)

-- Parents --
Parent 1 (Set 1): [7, 1]
Parent 2 (Set 2): [5, 2]

## Crossover ##

-- Offspring --
Offsping 1: [5, 1]
Offsping 2: [7, 2]



### Mutation Operator

In [43]:
print("Before Mutation")
print(f"{'Offspring #':<1} | {'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*60)
print(f"{'Offspring 1':<1} | Store {offspring1[0]:<1} | {offspring1_coords[0][0]:<10.10f} | {offspring1_coords[0][1]:<10.10f}")
print(f"{'Offspring 1':<1} | Store {offspring1[1]:<1} | {offspring1_coords[1][0]:<10.10f} | {offspring1_coords[1][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[0]:<1} | {offspring2_coords[0][0]:<10.10f} | {offspring2_coords[0][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[1]:<1} | {offspring2_coords[1][0]:<10.10f} | {offspring2_coords[1][1]:<10.10f}")

# Randomly choose a mutation point
mutation_point = random.choice([0, 1])

# Executes the mutation
offspring1_coords[mutation_point] = offspring1_coords[mutation_point]+0.01
offspring2_coords[1 if mutation_point==0 else 0] = offspring2_coords[1 if mutation_point==0 else 0]+0.001

print("\n## Mutation ##")
print("Mutation at Gene index", mutation_point)

print("\nAfter Mutation")
print(f"{'Offspring #':<1} | {'Store #':<1} | {'Latitude':<12} | {'Longitude':<15}")
print("-"*60)
print(f"{'Offspring 1':<1} | Store {offspring1[0]:<1} | {offspring1_coords[0][0]:<10.10f} | {offspring1_coords[0][1]:<10.10f}")
print(f"{'Offspring 1':<1} | Store {offspring1[1]:<1} | {offspring1_coords[1][0]:<10.10f} | {offspring1_coords[1][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[0]:<1} | {offspring2_coords[0][0]:<10.10f} | {offspring2_coords[0][1]:<10.10f}")
print(f"{'Offspring 2':<1} | Store {offspring2[1]:<1} | {offspring2_coords[1][0]:<10.10f} | {offspring2_coords[1][1]:<10.10f}")

Before Mutation
Offspring # | Store # | Latitude     | Longitude      
------------------------------------------------------------
Offspring 1 | Store 5 | 7.0792195616 | 125.6250239650
Offspring 1 | Store 1 | 7.0901382366 | 125.6067659888
Offspring 2 | Store 7 | 7.0656564747 | 125.6106411320
Offspring 2 | Store 2 | 7.0887236013 | 125.6308328384

## Mutation ##
Mutation at Gene index 0

After Mutation
Offspring # | Store # | Latitude     | Longitude      
------------------------------------------------------------
Offspring 1 | Store 5 | 7.0892195616 | 125.6350239650
Offspring 1 | Store 1 | 7.0901382366 | 125.6067659888
Offspring 2 | Store 7 | 7.0656564747 | 125.6106411320
Offspring 2 | Store 2 | 7.0897236013 | 125.6318328384


### Offspring Evaluation and Comparison
We will now calculate the maximum distance of each of the offspring to their furthest client. First, we create another tempororary distance matrix for easy comparison.

In [44]:
# Compute the pairwise distance matrices for the mutated coordinates
offspring1_dist = cdist(client_coords, offspring1_coords, metric='euclidean')
offspring2_dist = cdist(client_coords, offspring2_coords, metric='euclidean')

# Build the labeled DataFrames (Rows = Clients, Columns = Store IDs)
offspring1_matrix = pd.DataFrame(
    offspring1_dist,
    index=clients.index,  # Keeps rows labeled as client1 to client5
    columns=offspring1    # Sets columns to the active store IDs (e.g., [3, 7])
)

offspring2_matrix = pd.DataFrame(
    offspring2_dist,
    index=clients.index,  # Keeps rows labeled as client1 to client5
    columns=offspring2    # Sets columns to the active store IDs (e.g., [1, 4])
)

In [45]:
# Evaluating first and second set using mini distance matrices
"""
Stores the distances. Format: "Client 1": [<Distance>, <Closest Store>]
- Distance is in degrees
- Closest Store is either 0 if closest to the first store in the set, 1 if otherwise
"""
of1_dists = {}
of2_dists = {}

for client in range(1, 5+1):
    of1store1 = offspring1[0]    # index of store1 in set1
    of1store2 = offspring1[1]    # index of store2 in set1
    of2store1 = offspring2[0]    # index of store1 in set2
    of2store2 = offspring2[1]    # index of store2 in set2

    s1distto_s1 = offspring1_matrix[of1store1][client] # dist of client to store1,set1
    s1distto_s2 = offspring1_matrix[of1store2][client] # dist of client to store2,set1

    s2distto_s1 = offspring2_matrix[of2store1][client] # dist of client to store1,set2
    s2distto_s2 = offspring2_matrix[of2store2][client] # dist of client to store2,set2

    # find the minimum among the two distances for Set 1
    if s1distto_s1 > s1distto_s2:
        of1_dists[f"client{client}"] = (s1distto_s2, 1)
    else:
        of1_dists[f"client{client}"] = (s1distto_s1, 0)

    # find the minimum among the two distances for Set 2
    if s2distto_s1 > s2distto_s2:
        of2_dists[f"client{client}"] = (s2distto_s2, 1)
    else:
        of2_dists[f"client{client}"] = (s2distto_s1, 0)

In [46]:
# Print offspring 1 results
print(f"{'Client':<10} | {'Distance':<12} | {'Store':<10}")
print("-" * 40)

for key, value in of1_dists.items():
    store_name = offspring1[value[1]]
    print(f"{key:<10} | {value[0]:<12.8f} | {store_name:<10}")

Client     | Distance     | Store     
----------------------------------------
client1    | 0.01344014   | 5         
client2    | 0.01163590   | 1         
client3    | 0.01306169   | 1         
client4    | 0.00499704   | 1         
client5    | 0.01760628   | 1         


In [47]:
# Print offspring 2 results
print(f"{'Client':<10} | {'Distance':<12} | {'Store':<10}")
print("-" * 40)

for key, value in of2_dists.items():
    store_name = offspring2[value[1]]
    print(f"{key:<10} | {value[0]:<12.8f} | {store_name:<10}")

Client     | Distance     | Store     
----------------------------------------
client1    | 0.01045387   | 2         
client2    | 0.01759805   | 2         
client3    | 0.01297810   | 7         
client4    | 0.01981644   | 7         
client5    | 0.00773387   | 7         


In [48]:
# Evaluate which set is the best
if max(of1_dists.values())[0] > max(of2_dists.values())[0]:
    best_of = offspring2
    print("The best set is Set 2:", offspring2, "The max distance (deg) is", max(of2_dists.values())[0])
else:
    best_of = offspring1
    print("The best set is Set 1:", offspring1, "The max distance (deg) is", max(of1_dists.values())[0])

The best set is Set 1: [5, 1] The max distance (deg) is 0.017606276304107264


In [49]:
# Initialize map center (Somewhere in downtown Davao City)
mymap = folium.Map(location=[7.080506387226319, 125.6120560701777],
                   zoom_start=15)

# dictionary of client coordinates
clients_map = {}
for _ in range(len(client_coords)):
    clients_map[f"Client {_+1}"] = [client_coords[_][0], client_coords[_][1]]

best_of_coords = offspring2_coords if best_of == offspring2 else offspring1_coords

# dictionary of store coordinates in the best set
best_set = {}
for _ in range(2):
    best_set[f"Mutated Store {best_of[_]}"] = [best_of_coords[_][0],
                                               best_of_coords[_][1]]

# to determine the max radius in the best set
best_dists = of2_dists if best_of == offspring2 else of1_dists
max_radius_degrees2 = max(best_dists.values())[0]

# Draws the clients on the map
for client, coords in clients_map.items():
    folium.Marker(
        location=coords,
        popup=client,
        icon=folium.Icon(color="blue", icon="person", prefix='fa')
    ).add_to(mymap)

# Draws the stores on the map
for store, coords in best_set.items():
    folium.Marker(
        location=coords,
        popup=store,
        icon=folium.Icon(color="red", icon="store", prefix='fa')
    ).add_to(mymap)

    # Draws the circles around the stores of the best set
    folium.Circle(
        location=coords,
        radius=max_radius_degrees2 * 111320, # Radius from deg to meters
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.2,
        popup="Coverage Area"
    ).add_to(mymap)

mymap

In [50]:
print("-- COMPARISON --")
print("Did the new solution (offspring) produce a better result?", max_radius_degrees1 > max_radius_degrees2)

print("\nParents fitness value:", max_radius_degrees1)
print("Offspring fitness value:", max_radius_degrees2)

-- COMPARISON --
Did the new solution (offspring) produce a better result? False

Parents fitness value: 0.015430246996441958
Offspring fitness value: 0.017606276304107264


> If elitism is applied, will any of the offspring replace any of the parents Support your answer with an explanation why you will replace or why
you will not replace them.

If an offspring's mutated layout achieves a smaller maximum bottleneck distance than both original parents, it represents a mathematically better spatial configuration and will replace a parent; but if both offspring result in larger bottleneck distances, neither will replace the parents because elitism discards poorer random mutations to safeguard our optimal baseline. In a real-world scenario such as this one, while the mutated coordinates do not align with the given 7 concrete physical locations, a winning offspring provides valuable insight by proving the initial fixed sites are sub-optimal and indicating that finding a new location just a few hundred meters away would mathematically optimize client coverage.